# No-RAG Baseline Evaluation

This notebook runs the same benchmark questions without retrieval, saves the outputs in the same schema as the existing pipeline, and evaluates the generated answers with the current `evaluation.py` script.

Notes:
- Output file: `inputs_and_outputs/output_no_rag.json`
- Benchmark file: `data/latest_benchmark_123_aligned_top5.json`
- Retrieval metrics printed by `evaluation.py` are not meaningful for this notebook because `retrieved_context` is intentionally empty.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

from evaluation import evaluate_rag_pipeline
import llm_n_prompt
from llm_n_prompt import QwenLLM

PROJECT_ROOT = Path.cwd()
BENCHMARK_PATH = PROJECT_ROOT / "data" / "Super_final.json"
OUTPUT_PATH = PROJECT_ROOT / "inputs_and_outputs" / "output_no_rag.json"

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")
print(f"Benchmark    : {BENCHMARK_PATH}")
print(f"Output       : {OUTPUT_PATH}")
print(f"LLM model    : {llm_n_prompt.MODEL_NAME}")

In [ ]:
print(f"Loading model: {llm_n_prompt.MODEL_NAME}")
llm = QwenLLM(device=DEVICE)

def build_no_rag_messages(question: str):
    system = """You are ChefBot, an expert assistant specialising in South Asian cuisine.
Answer the user's question directly using your own model knowledge.
Be factual, concise, and avoid repetition.
If you are uncertain, say so briefly instead of inventing details."""

    user = f"""QUESTION: {question}

ANSWER:"""

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

def generate_no_rag_answer(question: str, max_new_tokens: int = 256) -> str:
    messages = build_no_rag_messages(question)
    return llm.generate(messages, max_tokens=max_new_tokens)

print("Model loaded. No-RAG baseline now uses the same QwenLLM generation path as the RAG pipeline.")

In [ ]:
with BENCHMARK_PATH.open("r", encoding="utf-8") as f:
    benchmark = json.load(f)
results = []

for item in tqdm(benchmark, desc="Running no-RAG baseline"):
    question = item["query"].strip()
    answer = generate_no_rag_answer(question)
    results.append({
        "query_id": str(item["id"] - 1),
        "query": question,
        "response": answer,
        "retrieved_context": [],
    })

payload = {"results": results}
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} results to {OUTPUT_PATH}")
pd.DataFrame(results).head()

In [ ]:
df = evaluate_rag_pipeline(str(OUTPUT_PATH), str(BENCHMARK_PATH))

summary = pd.DataFrame([
    {
        "run": "No RAG baseline",
        "BLEU (nltk)": df["bleu_nltk"].mean(),
        "BERTScore F1": df["bert_f1"].mean(),
    }
])
summary